### Step 0:

In [ ]:
import os, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.io as sio
from scipy.signal import butter, sosfiltfilt, lfilter, welch, filtfilt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

import mne
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dropout, Dense
from tensorflow.keras.layers import Input, BatchNormalization, GlobalAveragePooling1D, Bidirectional
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)


### Step 2:

In [ ]:
folder1 = Path(r"C:\Users\Cyan\Desktop\DataScience_Notes\DataScience_Capstone\EEG Data")
folder2 = Path(r"C:\Users\Cyan\Desktop\DataScience_Notes\DataScience_Capstone\eeg-during-mental-arithmetic-tasks-1.0.0")

all_files = [str(f) for f in folder1.glob("*.mat")] + [str(f) for f in folder1.glob("*.edf")]
all_files += [str(f) for f in folder2.glob("*.mat")] + [str(f) for f in folder2.glob("*.edf")]

print("Found", len(all_files), "files")

def load_mat_file(path):
    mat = sio.loadmat(path)
    o_struct = mat['o'][0,0]
    eeg_data = np.asarray(o_struct['data'])
    fs = int(o_struct['sampFreq'][0,0])
    subject_id = o_struct['id'][0]
    markers = o_struct['marker'].flatten()
    trials = o_struct['trials']
    return eeg_data, fs, subject_id, markers, trials

# 
fname = all_files[0]   # or any index, e.g. all_files[5]

# Load depending on file type
if fname.endswith(".mat"):
    eeg_data, fs, subject_id, markers, trials = load_mat_file(fname)
    ch = eeg_data[:,0]        # first channel as signal array
    sf = fs                   # sampling frequency (int)
elif fname.endswith(".edf"):
    raw = mne.io.read_raw_edf(fname, preload=True, verbose=False)
    data, sf = raw.get_data(), int(raw.info['sfreq'])
    ch = data[0]              # first channel as signal array
else:
    raise ValueError("Unsupported file format")





In [ ]:

#
fname = r"C:\Users\Cyan\Desktop\DataScience_Notes\DataScience_Capstone\EEG Data\eeg_record5.mat"
eeg_data, fs, subject_id, markers, trials = load_mat_file(fname)

print("EEG shape:", eeg_data.shape)
print("Sampling frequency:", fs)
print("Subject ID:", subject_id)
print("Markers shape:", markers.shape)
print("Trials shape:", trials.shape)

# Plot stacked EEG channels (hospital-style view)
n_samples, n_channels = eeg_data.shape
time = np.arange(n_samples) / fs
offset = 100

plt.figure(figsize=(15, 10))
for ch in range(n_channels):
    plt.plot(time, eeg_data[:, ch] + ch * offset, color='black')
    plt.text(-0.5, ch * offset, f"Ch {ch+1}", va='center')
plt.title(f"Raw EEG - Subject {subject_id}")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude + offset")
plt.yticks([])
plt.show()

# Plot first 1000 samples of channel 0
plt.figure(figsize=(12,4))
plt.plot(time[:1000], eeg_data[:1000, 0], color='tab:blue')
plt.title(f"EEG Signal (Subject {subject_id}, Channel 0)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()


In [ ]:
# Visualizing edf files

# Path to the EDF file
fname = r"C:\Users\Cyan\Desktop\DataScience_Notes\DataScience_Capstone\eeg-during-mental-arithmetic-tasks-1.0.0\Subject00_1.edf"


# For .edf files
raw = mne.io.read_raw_edf(fname, preload=True, verbose=False)
data, sf = raw.get_data(), int(raw.info['sfreq'])
ch = data[0]              # first channel as signal array

In [ ]:

# Extract data and metadata
data = raw.get_data()            # shape: (n_channels, n_samples)
sf = int(raw.info['sfreq'])      # sampling frequency
ch_names = raw.ch_names

print("Channels:", len(ch_names))
print("Sampling frequency:", sf)
print("Data shape:", data.shape)

# Plot first channel (first 10 seconds)
seconds_to_plot = 10
ch0 = data[0]
n_plot = min(len(ch0), int(seconds_to_plot * sf))
t = np.arange(n_plot) / sf

plt.figure(figsize=(12,4))
plt.plot(t, ch0[:n_plot], color='tab:blue')
plt.title(f"{Path(fname).name} — {ch_names[0]} — first {seconds_to_plot}s")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()

# Hospital-style stacked plot of all channels
plt.figure(figsize=(15,10))
offset = 100
for i, ch in enumerate(data):
    plt.plot(t, ch[:n_plot] + i*offset, color='black')
    plt.text(-0.5, i*offset, ch_names[i], va='center')
plt.title("EDF EEG — stacked channels view")
plt.xlabel("Time (s)")
plt.yticks([])
plt.show()


### Step 3:

In [ ]:
# Shuffle files
random.shuffle(all_files)

# Define split percentages
train_pct, val_pct, test_pct = 0.70, 0.15, 0.15
n_total = len(all_files)

# Compute counts
n_train = int(train_pct * n_total)
n_val   = int(val_pct * n_total)
n_test  = n_total - n_train - n_val   # ensures total consistency

# Split dataset
train_files = all_files[:n_train]
val_files   = all_files[n_train:n_train+n_val]
test_files  = all_files[n_train+n_val:]


### Step 4:

In [ ]:
def bandpass_filter(signal, sf, low=1, high=50, order=4):
    sos = butter(order, [low, high], btype='band', fs=sf, output='sos')
    return sosfiltfilt(sos, signal)

def segment_signal(signal, sf, window_sec=2, step_sec=1):
    epoch_len, step = int(window_sec*sf), int(step_sec*sf)
    return [signal[i:i+epoch_len] for i in range(0, len(signal)-epoch_len+1, step)]

def normalize_epoch(epoch):
    return StandardScaler().fit_transform(epoch.reshape(-1,1)).flatten()


In [ ]:
# Ensure ch is the EEG signal array and sf is the sampling frequency (int)
# Example: ch = eeg_data[:,0], sf = fs

# Apply bandpass filter first
ch_filtered = bandpass_filter(ch, sf)

# Slice the first N samples safely (up to 10 seconds)
N = min(len(ch), 10 * sf)
raw_segment = ch[:N]
filtered_segment = ch_filtered[:N]

# Time axis
t = np.arange(N) / sf

# Plot raw vs filtered
plt.figure(figsize=(12,8))
plt.plot(t, raw_segment, label='Raw', alpha=0.6)
plt.plot(t, filtered_segment, label='Filtered', alpha=0.9)
plt.title("Raw vs Bandpass Filtered Signal (first 10 seconds)")
plt.xlabel("Time (s)")
plt.legend()
plt.show()


### Step 5: Feature Extraction

In [ ]:
import numpy as np
from scipy.signal import welch

def hjorth_params(signal):
    """
    Hjorth parameters: Activity, Mobility, Complexity
    """
    activity = np.var(signal)

    diff1 = np.diff(signal)
    var_diff1 = np.var(diff1)
    mobility = np.sqrt(var_diff1 / activity) if activity > 0 else 0

    diff2 = np.diff(diff1)
    var_diff2 = np.var(diff2)
    mobility_diff1 = np.sqrt(var_diff2 / var_diff1) if var_diff1 > 0 else 0
    complexity = mobility_diff1 / mobility if mobility > 0 else 0

    return activity, mobility, complexity

def bandpower(epoch, sf, band, window_sec=None):
    low, high = band
    nperseg = window_sec * sf if window_sec else sf * 2
    freqs, psd = welch(epoch, sf, nperseg=nperseg)
    freq_res = freqs[1] - freqs[0]
    idx_band = np.logical_and(freqs >= low, freqs <= high)
    return np.sum(psd[idx_band]) * freq_res

def spectral_entropy(epoch, sf, normalize=True):
    freqs, psd = welch(epoch, sf, nperseg=sf*2)
    psd_norm = psd / np.sum(psd)
    entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-12))
    if normalize:
        entropy /= np.log2(len(psd_norm))
    return entropy

def extract_features(epoch, sf):
    hj_a, hj_m, hj_c = hjorth_params(epoch)
    return {
        "theta": bandpower(epoch, sf, (4,7)),
        "alpha": bandpower(epoch, sf, (8,12)),
        "beta":  bandpower(epoch, sf, (13,30)),
        "gamma": bandpower(epoch, sf, (30,45)),
        "mean":  float(np.mean(epoch)),
        "var":   float(np.var(epoch)),
        "hjorth_activity": hj_a,
        "hjorth_mobility": hj_m,
        "hjorth_complexity": hj_c,
        "spectral_entropy": spectral_entropy(epoch, sf)
    }


In [ ]:
# Take one epoch from your segmented signal
epochs = segment_signal(ch, sf, window_sec=2, step_sec=1)

if len(epochs) > 0:
    epoch = epochs[0]  # pick the first epoch

    # Compute PSD
    freqs, psd = welch(epoch, sf, nperseg=min(len(epoch), sf*2))

    # Plot PSD
    plt.figure(figsize=(10,4))
    plt.semilogy(freqs, psd, color='tab:green')
    plt.title("Power Spectral Density (one epoch)")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Power")
    plt.show()

    # Extract features
    base_feats = extract_features(epoch, sf)

    # Plot extracted features
    # Extract features
    base_feats = extract_features(epoch, sf)

    # Plot extracted features
    bands = list(base_feats.keys())
    vals = list(base_feats.values())
    plt.figure(figsize=(7,4))
    plt.bar(bands, vals, color='tab:orange')
    plt.title("Extracted Features (one epoch)")
    plt.ylabel("Value")
    plt.show()


### Step 6: Rule-based labeling

In [ ]:
def compute_thresholds(X):
    """
    Compute thresholds for each feature across all epochs.
    For simplicity, use the median as a decision boundary.
    """
    thresholds = {}
    feature_names = [
        "theta","alpha","beta","gamma",
        "mean","var","hjorth_activity",
        "hjorth_mobility","hjorth_complexity",
        "spectral_entropy",
        "theta_alpha","beta_alpha","gamma_beta","theta_beta"
    ]
    for i, name in enumerate(feature_names):
        thresholds[name] = np.median(X[:, i])
    return thresholds


def map_condition_rule(features, thresholds):
    """
    Map extracted features (including ratios) to a condition label.
    Returns: 0=Focused, 1=Distracted, 2=Fatigued, 3=Stressed, 4=Other
    """
    # Base features
    theta, alpha, beta, gamma, mean, var, hj_a, hj_m, hj_c, spec_ent = features[:10]
    # Band ratios
    theta_alpha, beta_alpha, gamma_beta, theta_beta = features[10:14]

    # Rule logic using ratios + base features
    if theta_alpha > thresholds["theta_alpha"]:
        return 2  # Fatigued
    elif beta_alpha > thresholds["beta_alpha"]:
        return 1  # Distracted
    elif alpha > thresholds["alpha"] and beta < thresholds["beta"]:
        return 0  # Focused
    elif spec_ent > thresholds["spectral_entropy"] or gamma_beta > thresholds["gamma_beta"]:
        return 3  # Stressed
    else:
        return 4  # Other


In [ ]:
# Segment the signal into epochs
epochs = segment_signal(ch, sf, window_sec=2, step_sec=1)

# Compute thresholds once from all epochs (with ratios included)
X_tmp = [list(extract_features_with_ratios(ep, sf).values()) for ep in epochs]
thresholds = compute_thresholds(np.array(X_tmp))

# Generate rule-based labels
rule_labels = [map_condition_rule(list(extract_features_with_ratios(ep, sf).values()), thresholds) 
               for ep in epochs]


# Plot labels across epochs
plt.figure(figsize=(12,2))
plt.plot(rule_labels, marker='o', linestyle='-', color='tab:purple')
plt.title("Rule-based Condition Labels Across Epochs")
plt.xlabel("Epoch index")
plt.yticks([0,1,2,3,4], ["Focused","Distracted","Fatigued","Stressed","Other"])
plt.show()


### Step 7: Baseline modelling- Randomforest Dataset building

In [ ]:

def build_dataset(files, window_sec=2, step_sec=1):
    X, y = [], []
    for fname in files:
        # Load and filter EEG signal
        if fname.endswith(".edf"):
            raw = mne.io.read_raw_edf(fname, preload=True, verbose=False)
            data, sf = raw.get_data(), int(raw.info['sfreq'])
            ch = bandpass_filter(data[0], sf)
        elif fname.endswith(".mat"):
            eeg_data, fs, subject_id, markers, trials = load_mat_file(fname)
            ch, sf = bandpass_filter(eeg_data[:,0], fs), fs
        else:
            continue

        # Segment into epochs
        epochs = segment_signal(ch, sf, window_sec=window_sec, step_sec=step_sec)

        # Compute thresholds once per file
        X_tmp = [list(extract_features_with_ratios(ep, sf).values()) for ep in epochs]
        thresholds = compute_thresholds(np.array(X_tmp))

        # Extract features + labels
        for ep in epochs:
            feats = extract_features_with_ratios(ep, sf)
            X.append(list(feats.values()))
            y.append(map_condition_rule(list(feats.values()), thresholds))
    
    return np.array(X), np.array(y)


In [ ]:
from imblearn.over_sampling import SMOTE
import numpy as np

def augment_epochs(epochs, sf, jitter_std=0.01):
    """
    Apply jitter augmentation to EEG epochs.
    jitter_std: standard deviation of Gaussian noise relative to signal variance
    """
    augmented = []
    for ep in epochs:
        noise = np.random.normal(0, jitter_std * np.std(ep), size=len(ep))
        augmented.append(ep + noise)
    return augmented

def build_balanced_dataset(files, window_sec=2, step_sec=1, use_smote=True, jitter=True):
    X, y = [], []
    for fname in files:
        # Load and filter EEG signal
        if fname.endswith(".edf"):
            raw = mne.io.read_raw_edf(fname, preload=True, verbose=False)
            data, sf = raw.get_data(), int(raw.info['sfreq'])
            ch = bandpass_filter(data[0], sf)
        elif fname.endswith(".mat"):
            eeg_data, fs, subject_id, markers, trials = load_mat_file(fname)
            ch, sf = bandpass_filter(eeg_data[:,0], fs), fs
        else:
            continue

        # Segment into epochs
        epochs = segment_signal(ch, sf, window_sec=window_sec, step_sec=step_sec)

        # Augment minority classes with jitter
        if jitter:
            epochs += augment_epochs(epochs, sf)

        # Compute thresholds once per file
        X_tmp = [list(extract_features_with_ratios(ep, sf).values()) for ep in epochs]
        thresholds = compute_thresholds(np.array(X_tmp))

        # Extract features + labels
        for ep in epochs:
            feats = extract_features_with_ratios(ep, sf)
            X.append(list(feats.values()))
            y.append(map_condition_rule(list(feats.values()), thresholds))

    X, y = np.array(X), np.array(y)

    # Apply SMOTE in feature space
    if use_smote:
        sm = SMOTE(random_state=42)
        X, y = sm.fit_resample(X, y)

    return X, y


In [ ]:
# Step 7.1: Balanced dataset with augmentation
X_all, y_all = build_balanced_dataset(all_files, window_sec=2, step_sec=1,
                                      use_smote=True, jitter=True)

# Encode labels
encoder = LabelEncoder()
y_enc = encoder.fit_transform(y_all)
num_classes = len(np.unique(y_enc))
y_cat = to_categorical(y_enc, num_classes=num_classes)

# Train/test split
X_train, X_test, y_train_enc, y_test_enc = train_test_split(
    X_all, y_enc, test_size=0.3, random_state=42
)

# Prepare inputs for RF and DL
X_train_rf, X_test_rf = X_train, X_test
y_train_rf, y_test_rf = y_train_enc, y_test_enc

X_train_dl = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_dl  = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
y_train_dl = to_categorical(y_train_enc, num_classes=num_classes)
y_test_dl  = to_categorical(y_test_enc, num_classes=num_classes)


In [ ]:
# Train RandomForest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_rf, y_train_rf)

Deep Learning 

Training Deep Learning models before class weights & Balancing

In [ ]:
#Train Raw CNN
X_train_dl = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_dl  = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
y_train_dl = to_categorical(y_train_enc, num_classes=num_classes)
y_test_dl  = to_categorical(y_test_enc, num_classes=num_classes)

cnn_model = Sequential([
    Input(shape=(X_train.shape[1], 1)),
    Conv1D(32, 3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(2),
    Conv1D(64, 3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.4),
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history_cnn = cnn_model.fit(X_train_dl, y_train_dl, epochs=30, batch_size=32,
                            validation_data=(X_test_dl, y_test_dl), verbose=1)


In [ ]:
#Train Raw LSTM
lstm_model = Sequential([
    Input(shape=(X_train.shape[1], 1)),
    Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3)),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history_lstm = lstm_model.fit(X_train_dl, y_train_dl, epochs=30, batch_size=32,
                              validation_data=(X_test_dl, y_test_dl), verbose=1)


In [ ]:
#Train Raw CNN+BiLSTM 
model = Sequential([
    Input(shape=(X_train.shape[1], 1)),
    Conv1D(32, 3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(2),
    Conv1D(64, 3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(2),
    Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3)),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(X_train_dl, y_train_dl, epochs=30, batch_size=32,
                    validation_data=(X_test_dl, y_test_dl),
                    callbacks=[ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)],
                    verbose=1)


Balancing the dataset

In [ ]:

# Balance dataset

X_bal, y_bal = build_dataset(train_files, window_sec=2, step_sec=1)


# One-hot encode labels
num_classes = len(np.unique(y_bal))
y_bal_cat = to_categorical(y_bal, num_classes=num_classes)

# Train/test split
from sklearn.model_selection import train_test_split

X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_bal, y_bal_cat, test_size=0.3, random_state=42
)
# Reshape for CNN/LSTM input
X_train_bal = X_train_bal.reshape((X_train_bal.shape[0], X_train_bal.shape[1], 1))
X_test_bal  = X_test_bal.reshape((X_test_bal.shape[0], X_test_bal.shape[1], 1))


Step 9: Models-Enhanced

CNN‑only

In [ ]:
cnn_model = Sequential([
    Input(shape=(X_train_bal.shape[1], 1)),
    Conv1D(32, kernel_size=5, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(2),
    Conv1D(64, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.5),
    GlobalAveragePooling1D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])


Lstm only

In [ ]:

lstm_model = Sequential([
    Input(shape=(X_train_bal.shape[1], 1)),
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.4, recurrent_dropout=0.4)),
    Bidirectional(LSTM(32, dropout=0.3, recurrent_dropout=0.3)),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dense(num_classes, activation='softmax')
])



CNN+BiLSTM

In [ ]:
cnn_lstm_model = Sequential([
    Input(shape=(X_train_bal.shape[1], 1)),
    Conv1D(32, 5, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(2),
    Conv1D(64, 3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(2),
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.4, recurrent_dropout=0.4)),
    Bidirectional(LSTM(32, dropout=0.3, recurrent_dropout=0.3)),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dense(num_classes, activation='softmax')
])


In [ ]:
mlp = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(5, activation='softmax')
])



In [ ]:
# Class weights
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_bal)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_bal)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}


In [ ]:
# Focal loss
import tensorflow as tf

def focal_loss(gamma=2., alpha=1.):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1-1e-7)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.math.pow(1 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * cross_entropy, axis=1))
    return loss


In [ ]:
# Training all DL models
cnn_model.compile(optimizer='adam', loss=focal_loss(), metrics=['accuracy'])
lstm_model.compile(optimizer='adam', loss=focal_loss(), metrics=['accuracy'])
cnn_lstm_model.compile(optimizer='adam', loss=focal_loss(), metrics=['accuracy'])
mlp.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
mlp.summary()



history_cnn = cnn_model.fit(X_train_bal, y_train_bal,
                            epochs=30, batch_size=32,
                            validation_data=(X_test_bal, y_test_bal),
                            class_weight=class_weight_dict,
                            verbose=1)

history_lstm = lstm_model.fit(X_train_bal, y_train_bal,
                              epochs=30, batch_size=32,
                              validation_data=(X_test_bal, y_test_bal),
                              class_weight=class_weight_dict,
                              verbose=1)

history_cnn_lstm = cnn_lstm_model.fit(X_train_bal, y_train_bal,
                                      epochs=30, batch_size=32,
                                      validation_data=(X_test_bal, y_test_bal),
                                      class_weight=class_weight_dict,
                                      verbose=1)

history_mlp = mlp.fit(X_train, y_train, epochs=20, batch_size=32,
                      validation_data=(X_test, y_test), verbose=1)

In [ ]:
from sklearn.metrics import f1_score

def compute_model_weights(models, X_test_rf, X_test_dl, y_true, rf_model=None):
    """
    Compute per-class weights for each model based on F1 scores.
    """
    num_classes = len(np.unique(y_true))
    weights = {}

    # RF model (use same test set indices as y_true)
    if rf_model is not None:
        # IMPORTANT: y_true must match X_test_rf length
        y_pred_rf = rf_model.predict(X_test_rf[:len(y_true)])
        f1_rf = f1_score(y_true, y_pred_rf, average=None, zero_division=0)
        weights["rf"] = f1_rf / f1_rf.sum()

    # Keras models
    for i, model in enumerate(models):
        y_pred_keras = model.predict(X_test_dl).argmax(axis=1)
        # Align lengths
        y_pred_keras = y_pred_keras[:len(y_true)]
        f1_keras = f1_score(y_true, y_pred_keras, average=None, zero_division=0)
        weights[f"keras{i}"] = f1_keras / f1_keras.sum()

    return weights


Evaluation & Error Analysis

In [ ]:
labels = ["Focused","Distracted","Fatigued","Stressed","Other"]

# Predictions
y_true = y_test_bal.argmax(axis=1)
y_pred_cnn = cnn_model.predict(X_test_bal).argmax(axis=1)
y_pred_lstm = lstm_model.predict(X_test_bal).argmax(axis=1)
y_pred_dl = cnn_lstm_model.predict(X_test_bal).argmax(axis=1)

# Reports
print("CNN Report:\n", classification_report(y_true, y_pred_cnn, target_names=labels))
print("LSTM Report:\n", classification_report(y_true, y_pred_lstm, target_names=labels))
print("CNN+BiLSTM Report:\n", classification_report(y_true, y_pred_dl, target_names=labels))

# Confusion matrices
fig, axes = plt.subplots(1,3, figsize=(18,5))
for ax, preds, title in zip(axes, [y_pred_cnn,y_pred_lstm,y_pred_dl], ["CNN","LSTM","CNN+BiLSTM"]):
    cm = confusion_matrix(y_true, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(f"Confusion Matrix — {title}")
plt.show()

# Error breakdown
def error_analysis(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred)
    errors = {}
    for i, label in enumerate(labels):
        total = cm[i].sum()
        correct = cm[i,i]
        errors[label] = {
            "total": total,
            "correct": correct,
            "misclassified": total - correct,
            "error_rate": (total - correct) / total if total > 0 else 0
        }
    return errors

print("CNN Errors:", error_analysis(y_true, y_pred_cnn, labels))
print("LSTM Errors:", error_analysis(y_true, y_pred_lstm, labels))
print("CNN+BiLSTM Errors:", error_analysis(y_true, y_pred_dl, labels))


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

def plot_roc_auc(models, X_test_dl, y_true, y_test_dl, labels):
    plt.figure(figsize=(10,7))
    for i, model in enumerate(models):
        y_prob = model.predict(X_test_dl)
        try:
            auc = roc_auc_score(y_test_dl, y_prob, average="macro", multi_class="ovr")
            fpr, tpr, _ = roc_curve(y_test_dl.ravel(), y_prob.ravel())
            plt.plot(fpr, tpr, label=f"Model {i} (AUC={auc:.2f})")
        except Exception as e:
            print(f"Skipping ROC for model {i}: {e}")
    plt.plot([0,1],[0,1],"k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Per-Class ROC-AUC Curves")
    plt.legend()
    plt.show()

# Example usage
plot_roc_auc([cnn_model, lstm_model, cnn_lstm_model], X_test_dl, y_true, y_test_dl, labels)


Evaluation after Improvement

### Error Analysis

In [ ]:
# Collect Predictions
y_true = y_test_bal.argmax(axis=1)
y_pred_cnn = cnn_model.predict(X_test_bal).argmax(axis=1)
y_pred_lstm = lstm_model.predict(X_test_bal).argmax(axis=1)
y_pred_dl = cnn_lstm_model.predict(X_test_bal).argmax(axis=1)


In [ ]:
# Classification Report
from sklearn.metrics import classification_report

labels = ["Focused","Distracted","Fatigued","Stressed","Other"]

print("CNN Report:\n", classification_report(y_true, y_pred_cnn, target_names=labels))
print("LSTM Report:\n", classification_report(y_true, y_pred_lstm, target_names=labels))
print("CNN+BiLSTM Report:\n", classification_report(y_true, y_pred_dl, target_names=labels))


In [ ]:
# Confusion matrices
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1,3, figsize=(18,5))
for ax, preds, title in zip(axes, [y_pred_cnn,y_pred_lstm,y_pred_dl], ["CNN","LSTM","CNN+BiLSTM"]):
    cm = confusion_matrix(y_true, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(f"Confusion Matrix — {title}")
plt.show()


In [ ]:
# Per class Error Analysis
import numpy as np

def error_analysis(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred)
    errors = {}
    for i, label in enumerate(labels):
        total = cm[i].sum()
        correct = cm[i,i]
        errors[label] = {
            "total": total,
            "correct": correct,
            "misclassified": total - correct,
            "error_rate": (total - correct) / total if total > 0 else 0
        }
    return errors

print("CNN Errors:", error_analysis(y_true, y_pred_cnn, labels))
print("LSTM Errors:", error_analysis(y_true, y_pred_lstm, labels))
print("CNN+BiLSTM Errors:", error_analysis(y_true, y_pred_dl, labels))


In [ ]:
# Inspect Misclassified Classes
mis_idx = np.where(y_true != y_pred_dl)[0]
print("misclassified indices:", mis_idx[:10])

for i in mis_idx[:10]:
    print(f"Index {i}: True={labels[y_true[i]]}, Predicted={labels[y_pred_dl[i]]}")



**Detailed error breakdown per class**

In [ ]:
# True vs Predicted Labels
labels = ["Focused","Distracted","Fatigued","Stressed","Other"]

for i in mis_idx[:10]:  # first 10 misclassified samples
    print(f"Index {i}: True={labels[y_true[i]]}, Predicted={labels[y_pred_dl[i]]}")


In [ ]:
# Confusion Breakdown
from collections import defaultdict

def misclassification_breakdown(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred)
    breakdown = defaultdict(list)
    for i, true_label in enumerate(labels):
        for j, pred_label in enumerate(labels):
            if i != j and cm[i,j] > 0:
                breakdown[true_label].append((pred_label, cm[i,j]))
    return breakdown

breakdown = misclassification_breakdown(y_true, y_pred_dl, labels)
for true_class, errors in breakdown.items():
    print(f"{true_class} misclassified as:")
    for pred_class, count in errors:
        print(f"  {pred_class}: {count} times")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_misclassification_bars(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred)
    n_classes = len(labels)

    # Build stacked bars: correct vs misclassified counts
    correct = [cm[i,i] for i in range(n_classes)]
    misclassified = [cm[i].sum() - cm[i,i] for i in range(n_classes)]

    x = np.arange(n_classes)

    plt.figure(figsize=(10,6))
    plt.bar(x, correct, label="Correct", color="seagreen")
    plt.bar(x, misclassified, bottom=correct, label="Misclassified", color="tomato")

    plt.xticks(x, labels)
    plt.ylabel("Number of Samples")
    plt.title("Per-Class Prediction Breakdown")
    plt.legend()
    plt.show()

# Example usage for CNN+BiLSTM
plot_misclassification_bars(y_true, y_pred_dl, labels)


In [ ]:
# Error rate visualization
errors = error_analysis(y_true, y_pred_dl, labels)

plt.figure(figsize=(8,5))
plt.bar(errors.keys(), [v["error_rate"] for v in errors.values()], color="red")
plt.title("Per-Class Error Rates (CNN+BiLSTM)")
plt.ylabel("Error Rate")
plt.show()


**Interpretation**

-*Fatigued → Focused: EEG theta/alpha increases overlap with relaxed/focused states.*

-*Distracted → Stressed: Beta/gamma increases overlap with stress features.*

-*Other absorbing errors: Catch‑all class when features don’t fit thresholds.*

**Actionable Fixes**

1. *Add band ratios (theta/alpha, beta/alpha) to separate Fatigued vs Focused.*

2. *Use data augmentation (noise, jitter, scaling) for minority classes.*

3. *Apply class weights + balanced subsampling together.*

4. *Consider ensemble voting (CNN + RF + LSTM) to reduce bias.*

In [ ]:
import numpy as np

def ensemble_voting(rf_model, keras_models, X_test_rf, X_test_dl, num_classes, weights=None):
    """
    Ensemble voting across RandomForest + Keras models.
    - rf_model: trained RandomForest
    - keras_models: list of trained Keras models
    - X_test_rf: 2D test data for RF
    - X_test_dl: 3D test data for Keras models
    - num_classes: number of target classes
    - weights: list of weights for each model (default equal)
    """
    preds = []

    # RandomForest probabilities
    rf_probs = rf_model.predict_proba(X_test_rf)
    if rf_probs.shape[1] != num_classes:
        padded = np.zeros((rf_probs.shape[0], num_classes))
        padded[:, :rf_probs.shape[1]] = rf_probs
        rf_probs = padded
    preds.append(rf_probs)

    # Keras models probabilities
    for model in keras_models:
        keras_probs = model.predict(X_test_dl)
        if keras_probs.shape[1] != num_classes:
            padded = np.zeros((keras_probs.shape[0], num_classes))
            padded[:, :keras_probs.shape[1]] = keras_probs
            keras_probs = padded
        preds.append(keras_probs)

    # Apply weights
    if weights is None:
        weights = [1/len(preds)] * len(preds)  # equal weights
    weighted_preds = np.zeros_like(preds[0])
    for w, p in zip(weights, preds):
        weighted_preds += w * p

    # Final prediction
    y_pred = np.argmax(weighted_preds, axis=1)
    return y_pred


In [ ]:
y_true = y_test.argmax(axis=1)

# Weighted ensemble: give CNN+BiLSTM more influence
weights = [0.1, 0.25, 0.25, 0.4]  # RF, CNN, LSTM, CNN+BiLSTM

y_true = y_test_dl.argmax(axis=1)

y_pred_ensemble = ensemble_voting(
    rf,
    [cnn_model, lstm_model, model],
    X_test_rf,   # 2D input for RF
    X_test_dl,   # 3D input for DL
    num_classes,
    weights=[0.1, 0.25, 0.25, 0.4]  # weighted voting
)

print("Ensemble Report:\n", classification_report(y_true, y_pred_ensemble, target_names=labels))


cm = confusion_matrix(y_true, y_pred_ensemble)
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix — Weighted Ensemble Voting")
plt.show()


Step 10: Adaptive Learning Engine

In [ ]:
def adaptive_engine(model, encoder, stream, sf=128, window_sec=2, step_sec=1):
    """
    Adaptive learning engine that adjusts study material based on EEG predictions.
    - model: trained CNN+LSTM
    - encoder: label encoder for states
    - stream: incoming EEG signal (numpy array or generator)
    """
    epoch_len = sf * window_sec
    step = sf * step_sec
    actions = []

    for start in range(0, len(stream) - epoch_len + 1, step):
        epoch = stream[start:start+epoch_len].reshape(1, epoch_len, 1)
        pred = model.predict(epoch).argmax(axis=1)[0]
        state = encoder.classes_[pred]

        # Map state to adaptive action
        if state == "Focused":
            action = "Maintain current difficulty"
        elif state == "Distracted":
            action = "Switch to interactive exercise"
        elif state == "Fatigued":
            action = "Suggest short break"
        elif state == "Stressed":
            action = "Offer calming material"
        else:
            action = "Neutral state — no change"

        actions.append((state, action))

    return actions


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

labels = ["Focused","Distracted","Fatigued","Stressed","Other"]

def evaluate_models(models, X_test, y_test, model_names, enhanced=False):
    """
    Evaluate multiple models and print classification reports, confusion matrices, and F1 scores.
    """
    y_true = y_test.argmax(axis=1) if y_test.ndim > 1 else y_test
    
    print("\n" + "="*50)
    print("Evaluation " + ("AFTER Enhancements" if enhanced else "BEFORE Enhancements"))
    print("="*50)
    
    fig, axes = plt.subplots(1, len(models), figsize=(6*len(models), 5))
    if len(models) == 1: axes = [axes]  # handle single model case
    
    for ax, model, name in zip(axes, models, model_names):
        # Predictions
        y_pred = model.predict(X_test)
        if y_pred.ndim > 1:  # keras softmax output
            y_pred = y_pred.argmax(axis=1)
        
        # Classification report
        print(f"\n{name} Report:\n", classification_report(y_true, y_pred, target_names=labels))
        
        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=labels, yticklabels=labels, ax=ax)
        ax.set_title(f"Confusion Matrix — {name}")
        
        # Macro F1
        f1 = f1_score(y_true, y_pred, average="macro")
        print(f"{name} Macro F1: {f1:.3f}")
    
    plt.tight_layout()
    plt.show()

# Example usage:
# Before enhancements
evaluate_models(
    models=[cnn_model, lstm_model, model],   # raw CNN, raw LSTM, raw CNN+BiLSTM
    X_test=X_test_dl,
    y_test=y_test_dl,
    model_names=["CNN (Raw)", "LSTM (Raw)", "CNN+BiLSTM (Raw)"],
    enhanced=False
)

# After enhancements
evaluate_models(
    models=[cnn_model, lstm_model, cnn_lstm_model, mlp],   # enhanced CNN, LSTM, CNN+BiLSTM, MLP
    X_test=X_test_bal,
    y_test=y_test_bal,
    model_names=["CNN (Enhanced)", "LSTM (Enhanced)", "CNN+BiLSTM (Enhanced)", "MLP"],
    enhanced=True
)


In [ ]:
# After training
model.save("cnn_lstm_model.h5")

import joblib
joblib.dump(encoder, "label_encoder.pkl")


**FEEDBACKS**
- *Why CNN+LSTM*
- *Show how they function separately*
- *Detailed Evaluations and Error analyses*
- *Solve the ignored classes(Fatigued & Distraction)*
- Model science-charts,flow of data(Data pipeline)
- Explainability model

Label Encoding

Purpose: Converts categorical labels (like "Focused", "Distracted", "Fatigued", etc.) into numerical values (0, 1, 2, ...).
Why it’s needed: Machine learning models (Random Forest, CNN, LSTM) can’t directly process text labels. They require numeric inputs.

- *Ensures consistency and allows one-hot encoding later for deep learning models.*

BatchNormalization

Purpose: Normalizes the activations of each layer during training.
How it works: It rescales and shifts outputs so they have a stable mean and variance.
Why it’s important:
- Prevents exploding/vanishing gradients.
- Speeds up convergence (training stabilizes faster).
- Acts as a regularizer, reducing overfitting.

*In my CNN/LSTM models: After convolution layers, BatchNormalization ensures that feature maps are well-scaled before pooling or dense layers.*

MaxPooling & GlobalAveragePooling
MaxPooling:

Purpose: Downsamples feature maps by taking the maximum value in a window (e.g., 2×2).
Effect: Reduces dimensionality, highlights the most dominant features, and adds translation invariance.

*In EEG context: Helps capture the strongest signal patterns (like bursts in theta or beta bands) while discarding noise.*

GlobalAveragePooling (GAP):
Purpose: Instead of flattening all features, GAP computes the average of each feature map.
Effect: Produces a compact representation with fewer parameters, reducing overfitting.

*Why it’s better than Flatten in some cases: GAP forces the model to focus on overall feature presence rather than exact positions, which is useful for EEG signals where patterns matter more than spatial location*

StandardScaler

- I applied StandardScaler().fit_transform(epoch.reshape(-1,1)) to each epoch.
This ensures every 2-second EEG slice has normalized values before feature extraction.

Result: cleaner inputs → more stable thresholds → better classification of states (Focused, Distracted, Fatigued, etc.).
->*StandardScaler makes EEG features comparable, stabilizes training, and prevents bias toward high-amplitude signals.*

One-Hot encoding in DeepLearning

*One-hot encoding ensures my CNN/LSTM models correctly learn to classify EEG states without introducing false numeric relationships between categories.*
Multi-class classification:  
Your model predicts states like "Focused", "Distracted", "Fatigued", "Stressed", "Other". These are categorical classes, not numeric values.

1. Neural network compatibility:  
Deep learning models (CNN, LSTM, CNN+BiLSTM) expect the target labels to be in a format that matches the output layer.
- If you have 5 classes, your final dense layer has 5 neurons with softmax activation.
- One-hot encoding ensures the target labels align with this structure.

2. Avoids ordinal bias:  
- If you used integers (0, 1, 2, 3, 4), the model might mistakenly treat them as ordered values (thinking "Fatigued = 2" is somehow closer to "Distracted = 1" than "Stressed = 3").
- One-hot encoding removes this false sense of order.

Next Steps:

Implement band ratios + augmentation.

Add attention layer to CNN+BiLSTM.

Enhance error analysis with ROC/PR curves.

Build a polished README with diagrams and recruiter‑friendly highlights.

Package the adaptive engine demo in Streamlit for interactive showcasing.